# Train Passenger Travel Pattern Analysis

**Student Name:** A.Jeevan  
**Roll Number:** 1601-25-737-912  
**Project:** Train Passenger Travel Pattern Analysis

## Objective
Analyze ticket volume, train-type and class preferences, popular routes, booking status, booking channels, passenger demographics, and time-based travel trends.

> **Dataset note:** This is a synthetic demonstration dataset created for academic practice. It does not contain real passenger or railway records.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the Dataset
Keep the CSV file in the same folder as this notebook.

In [ ]:
df = pd.read_csv("Train_Passenger_Travel_Pattern_Analysis_dataset.csv")
df.head()

## 3. Explore the Dataset

In [ ]:
print("Dataset shape:", df.shape)
df.info()
display(df.describe(include="all").T)
print("\nMissing values:")
display(df.isnull().sum())

## 4. Data Cleaning

In [ ]:
df["Journey_Date"] = pd.to_datetime(df["Journey_Date"], errors="coerce")
df["Ticket_Fare_INR"] = pd.to_numeric(df["Ticket_Fare_INR"], errors="coerce")
df = df.drop_duplicates(subset=["Ticket_ID"])
df = df.dropna(subset=["Journey_Date", "Ticket_Fare_INR", "Train_Type", "Booking_Status"])
df["Month"] = df["Journey_Date"].dt.to_period("M").astype(str)
df["Weekday"] = df["Journey_Date"].dt.day_name()
df["Hour"] = pd.to_datetime(df["Departure_Time"], format="%H:%M:%S", errors="coerce").dt.hour
df["Route"] = df["Source_City"] + " → " + df["Destination_City"]
print("Cleaned dataset shape:", df.shape)
df.head()

## 5. Summary Metrics

In [ ]:
total = len(df)
revenue = df["Ticket_Fare_INR"].sum()
avg = df["Ticket_Fare_INR"].mean()
confirmed = df["Booking_Status"].eq("Confirmed").mean() * 100
avg_dist = df["Distance_KM"].mean()
print(f"Total tickets booked: {total}")
print(f"Total fare value: ₹{revenue:,.2f}")
print(f"Average ticket fare: ₹{avg:,.2f}")
print(f"Average journey distance: {avg_dist:,.1f} km")
print(f"Confirmed booking rate: {confirmed:.2f}%")

## 6. Tickets by Train Type

In [ ]:
counts = df["Train_Type"].value_counts()
display(counts.to_frame("Ticket_Count"))
counts.plot(kind="bar", figsize=(8,4), title="Tickets by Train Type")
plt.xlabel("Train Type"); plt.ylabel("Number of Tickets")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## 7. Fare Value by Travel Class

In [ ]:
class_value = df.groupby("Travel_Class")["Ticket_Fare_INR"].sum().sort_values(ascending=False)
display(class_value.to_frame("Total_Fare_INR"))
class_value.plot(kind="bar", figsize=(8,4), title="Total Fare Value by Travel Class")
plt.xlabel("Travel Class"); plt.ylabel("Total Fare (INR)")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()
display(df.groupby("Travel_Class")["Ticket_Fare_INR"].mean().sort_values(ascending=False).to_frame("Average_Fare_INR"))

## 8. Most Popular Routes

In [ ]:
route_summary = df.groupby("Route").agg(
    Ticket_Count=("Ticket_ID","count"),
    Total_Fare_INR=("Ticket_Fare_INR","sum"),
    Average_Fare_INR=("Ticket_Fare_INR","mean")
).sort_values("Ticket_Count", ascending=False)
display(route_summary.head(10).round(2))
route_summary["Ticket_Count"].head(10).sort_values().plot(kind="barh", figsize=(9,5), title="Top 10 Routes by Ticket Count")
plt.xlabel("Number of Tickets"); plt.ylabel("Route"); plt.tight_layout(); plt.show()

## 9. Booking Status

In [ ]:
status_counts = df["Booking_Status"].value_counts()
display(status_counts.to_frame("Count"))
status_counts.plot(kind="pie", autopct="%1.1f%%", figsize=(6,6), title="Booking Status Distribution")
plt.ylabel(""); plt.tight_layout(); plt.show()

## 10. City-Level Analysis (Departures)

In [ ]:
city_summary = df.groupby("Source_City").agg(
    Ticket_Count=("Ticket_ID","count"),
    Total_Fare_INR=("Ticket_Fare_INR","sum"),
    Average_Fare_INR=("Ticket_Fare_INR","mean")
).sort_values("Ticket_Count", ascending=False)
display(city_summary.round(2))
city_summary["Ticket_Count"].plot(kind="bar", figsize=(8,4), title="Tickets by Departure City")
plt.xlabel("Departure City"); plt.ylabel("Number of Tickets"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()

## 11. Monthly Travel Trend

In [ ]:
monthly = df.groupby("Month").agg(
    Ticket_Count=("Ticket_ID","count"),
    Total_Fare_INR=("Ticket_Fare_INR","sum")
).sort_index()
display(monthly.round(2))
monthly["Ticket_Count"].plot(marker="o", figsize=(9,4), title="Monthly Ticket Volume")
plt.xlabel("Month"); plt.ylabel("Number of Tickets")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 12. Ticket Fare Distribution

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(data=df, x="Ticket_Fare_INR", bins=30, kde=True)
plt.title("Distribution of Ticket Fares")
plt.xlabel("Ticket Fare (INR)"); plt.tight_layout(); plt.show()

## 13. Travel Class vs Booking Status

In [ ]:
tab = pd.crosstab(df["Travel_Class"], df["Booking_Status"])
display(tab)
tab.plot(kind="bar", stacked=True, figsize=(9,4), title="Booking Status by Travel Class")
plt.xlabel("Travel Class"); plt.ylabel("Number of Tickets")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## 14. Departures by Hour

In [ ]:
hourly = df.groupby("Hour").size().reindex(range(24), fill_value=0)
hourly.plot(kind="line", marker="o", figsize=(9,4), title="Train Departures by Hour of Day")
plt.xlabel("Hour (24-hour clock)"); plt.ylabel("Number of Tickets")
plt.xticks(range(0,24,2)); plt.tight_layout(); plt.show()

## 15. Weekday vs Weekend

In [ ]:
df["Day_Type"] = np.where(df["Is_Weekend"].astype(str).str.lower().isin(["true","1"]), "Weekend", "Weekday")
day_summary = df.groupby("Day_Type").agg(
    Ticket_Count=("Ticket_ID","count"),
    Total_Fare_INR=("Ticket_Fare_INR","sum"),
    Average_Fare_INR=("Ticket_Fare_INR","mean")
)
display(day_summary.round(2))
day_summary["Ticket_Count"].plot(kind="bar", figsize=(6,4), title="Weekday vs Weekend Tickets")
plt.xlabel("Day Type"); plt.ylabel("Number of Tickets")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 16. Booking Mode and Travel Purpose

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
df["Booking_Mode"].value_counts().plot(kind="bar", ax=ax[0], title="Tickets by Booking Mode")
df["Travel_Purpose"].value_counts().plot(kind="bar", ax=ax[1], title="Tickets by Travel Purpose", color="seagreen")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 17. Passenger Age Group Analysis

In [ ]:
order = ["Under 18","18-25","26-40","41-60","60+"]
age_summary = df.groupby("Age_Group").agg(
    Ticket_Count=("Ticket_ID","count"),
    Average_Fare_INR=("Ticket_Fare_INR","mean"),
    Average_Distance_KM=("Distance_KM","mean")
).reindex(order)
display(age_summary.round(2))
age_summary["Ticket_Count"].plot(kind="bar", figsize=(7,4), title="Tickets by Passenger Age Group")
plt.xlabel("Age Group"); plt.ylabel("Number of Tickets"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## 18. Correlation Heatmap
Correlation does not prove causation.

In [ ]:
corr = df[["Ticket_Fare_INR","Distance_KM","Hour"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
sns.scatterplot(data=df, x="Distance_KM", y="Ticket_Fare_INR", hue="Travel_Class")
plt.title("Distance vs Ticket Fare"); plt.tight_layout(); plt.show()

## 19. Automatic Findings

In [ ]:
print("Most-used train type:", df["Train_Type"].value_counts().idxmax())
print("Travel class with highest total fare:", df.groupby("Travel_Class")["Ticket_Fare_INR"].sum().idxmax())
print("Most popular route:", df["Route"].value_counts().idxmax())
print("Busiest departure city:", df["Source_City"].value_counts().idxmax())
print("Peak departure hour:", df["Hour"].value_counts().idxmax())
print("Most common booking mode:", df["Booking_Mode"].value_counts().idxmax())
print("Cancelled tickets:", int(df["Booking_Status"].eq("Cancelled").sum()))
print("These findings describe only this synthetic sample.")

## 20. Conclusion
This project demonstrates how Python can analyze train passenger travel patterns using tables, summary statistics, and visualizations. It compares train types, travel classes, routes, booking outcomes, cities, passenger groups, and time patterns. Since the data is synthetic, findings must not be presented as real-world railway results.

## Dataset Dictionary
- `Ticket_ID`: unique demo ticket identifier
- `Journey_Date`, `Departure_Time`: date and time of departure
- `Passenger_ID`: synthetic passenger identifier
- `Source_City`, `Destination_City`: origin and destination
- `Distance_KM`: approximate journey distance
- `Train_Type`: Express, Superfast, Passenger, Vande Bharat, or Rajdhani
- `Travel_Class`: General, Sleeper, 3AC, 2AC, 1AC, Chair Car, or Executive Chair Car
- `Ticket_Fare_INR`: fare in Indian rupees
- `Booking_Status`: Confirmed, Waitlisted, or Cancelled
- `Booking_Mode`: Mobile App, Website, Station Counter, or Agent
- `Age_Group`: passenger age band
- `Travel_Purpose`: Work, Education, Leisure, Family Visit, or Medical
- `Is_Weekend`: whether the journey date falls on a weekend

**Student:** A.Jeevan  
**Roll Number:** 1601-25-737-912
